# ModernFinBERT v2 Stage 3: Long-Context Fine-Tuning

Load the Stage-2 model (trained on medium-context data in NB 01c) and fine-tune on long-context financial data.
- Stage-2 model: `neoyipeng/ModernFinBERT-v2-medium`
- Dataset: `neoyipeng/modernfinbert-training-v2-long` (28.8K train rows — earnings calls, SEC 10-K MD&A)
- Train subsample: 8000 rows (full set didn't finish in Kaggle's 12h cap at this context length)
- MAX_LENGTH = 4096 (operational ceiling — Stage 2 had 0% truncation here, Stage 3's prior 6144 had 52.8% truncation)
- Entity-aware input: `[CLS] entity [SEP] text [SEP]`
- Training target: `entity_sentiment`
- `logging_steps=5`, `save_strategy="steps"`, `save_steps=25` so a 12h kill leaves usable checkpoints + dense early-step diagnostics
- Evaluates on long, medium AND short test sets to check for regression
- Pushes final model to HuggingFace

Estimated time on T4: ~3-4 hours (batch=8 at 4096 tokens, 8K subsample, ~250 optimizer steps)

*Recipe: see [`notebooks/RECIPE.md`](RECIPE.md) for the full per-stage decision table.*

In [ ]:
%%capture
import os, re, torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer scikit-learn
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
%env UNSLOTH_DISABLE_FAST_GENERATION=1
%env TORCHDYNAMO_DISABLE=1

import torch
import torch._dynamo
torch._dynamo.config.disable = True

import numpy as np
from unsloth import FastModel, is_bfloat16_supported
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 3407
NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
LABEL_TO_ID = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
ID_TO_LABEL = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
MAX_LENGTH = 4096
TRAIN_SUBSAMPLE = 8000
STAGE2_MODEL = "neoyipeng/ModernFinBERT-v2-medium"
HF_PUSH_NAME = "neoyipeng/ModernFinBERT-v2"

cap = torch.cuda.get_device_capability()
assert cap[0] >= 7, f"GPU sm_{cap[0]}{cap[1]} not supported — need sm_70+ (T4/V100/A100)"
print(f"GPU: {torch.cuda.get_device_name()} (sm_{cap[0]}{cap[1]})")
print(f"Max tokens: {MAX_LENGTH}")

## 1. Load Long-Context Data

In [ ]:
ds_long = load_dataset("neoyipeng/modernfinbert-training-v2-long")
print("Long-context dataset:")
print(ds_long)

for split in ds_long:
    ds_long[split] = ds_long[split].select_columns(["text", "entity", "entity_sentiment"])

def map_labels(examples):
    examples["label"] = [LABEL_TO_ID[l] for l in examples["entity_sentiment"]]
    return examples

for split in ds_long:
    ds_long[split] = ds_long[split].map(map_labels, batched=True)
    ds_long[split] = ds_long[split].remove_columns(["entity_sentiment"])

# Subsample train so 1 epoch fits comfortably under Kaggle's 12h cap.
# Random sample preserves class proportions in expectation.
ds_long["train"] = ds_long["train"].shuffle(seed=SEED).select(range(TRAIN_SUBSAMPLE))
print(f"\nSubsampled train: {len(ds_long['train'])} rows")

print(f"\nLabel distribution (train, entity_sentiment):")
labels = ds_long["train"]["label"]
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name}: {labels.count(i)}")

entities = ds_long["train"]["entity"]
n_with_entity = sum(1 for e in entities if e not in ("NONE", "MARKET", "None", "", None))
print(f"\nEntity coverage: {n_with_entity}/{len(entities)} ({100*n_with_entity/len(entities):.1f}%)")

In [ ]:
# Also load short-context and medium-context TEST sets for regression check
ds_short_test = load_dataset("neoyipeng/modernfinbert-training-v2", split="test")
ds_short_test = ds_short_test.select_columns(["text", "entity", "entity_sentiment"])
ds_short_test = ds_short_test.map(map_labels, batched=True)
ds_short_test = ds_short_test.remove_columns(["entity_sentiment"])
print(f"Short-context test set: {len(ds_short_test)} rows (for regression check)")

ds_med_test = load_dataset("neoyipeng/modernfinbert-training-v2-medium", split="test")
ds_med_test = ds_med_test.select_columns(["text", "entity", "entity_sentiment"])
ds_med_test = ds_med_test.map(map_labels, batched=True)
ds_med_test = ds_med_test.remove_columns(["entity_sentiment"])
print(f"Medium-context test set: {len(ds_med_test)} rows (for regression check)")

## 2. Load Stage-2 Model + Fresh LoRA

In [ ]:
# T4 is sm_75 → no FlashAttention 2 support. Force SDPA to avoid the
# ModernBERT FA2 NaN-loss path (HF discussion #59, transformers#35988).
# Note: `reference_compile` defaults to is_flash_attn_2_available() which is
# False on T4, so we don't need to pass it — and unsloth forwards the kwarg
# to ModernBertForSequenceClassification.__init__, which rejects it.
model, tokenizer = FastModel.from_pretrained(
    model_name=STAGE2_MODEL,
    auto_model=AutoModelForSequenceClassification,
    max_seq_length=MAX_LENGTH,
    dtype=torch.float16,
    num_labels=NUM_CLASSES,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    load_in_4bit=False,
    attn_implementation="sdpa",
)
print(f"Loaded stage-2 model from {STAGE2_MODEL}")

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=32,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    task_type="SEQ_CLS",
)

In [ ]:
# Entity-aware tokenization: sentence pair format [CLS] entity [SEP] text [SEP]
def tokenize_function(examples):
    entities = [
        e if e not in ("NONE", "MARKET", "None", "", None) else ""
        for e in examples["entity"]
    ]
    return tokenizer(entities, examples["text"], truncation=True, max_length=MAX_LENGTH)

# Tokenize long-context train/val/test
tokenized_long = {}
for split in ds_long:
    tokenized_long[split] = ds_long[split].map(tokenize_function, batched=True, remove_columns=["text", "entity"])
    tokenized_long[split] = tokenized_long[split].rename_column("label", "labels")
    print(f"long/{split}: {len(tokenized_long[split])} rows tokenized")

# Tokenize short-context test set (for regression check)
tokenized_short_test = ds_short_test.map(tokenize_function, batched=True, remove_columns=["text", "entity"])
tokenized_short_test = tokenized_short_test.rename_column("label", "labels")
print(f"short/test: {len(tokenized_short_test)} rows tokenized")

# Tokenize medium-context test set (for regression check)
tokenized_med_test = ds_med_test.map(tokenize_function, batched=True, remove_columns=["text", "entity"])
tokenized_med_test = tokenized_med_test.rename_column("label", "labels")
print(f"medium/test: {len(tokenized_med_test)} rows tokenized")

# Token length stats for the long training data
lengths = [len(x) for x in tokenized_long["train"]["input_ids"]]
print(f"\nLong-context token lengths (train):")
print(f"  min={min(lengths)}, median={sorted(lengths)[len(lengths)//2]}, "
      f"max={max(lengths)}, mean={np.mean(lengths):.0f}")
n_truncated = sum(1 for l in lengths if l >= MAX_LENGTH)
trunc_pct = n_truncated / len(lengths)
print(f"  Truncated to {MAX_LENGTH}: {n_truncated} rows ({100*trunc_pct:.1f}%)")
assert trunc_pct < 0.10, f"Truncation rate {100*trunc_pct:.1f}% > 10% — adjust MAX_LENGTH or chunk pipeline"

## 3. Training

In [ ]:
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

def stratified_subset(ds, n_per_class=100, seed=SEED):
    rng = np.random.default_rng(seed)
    by_class = {c: [] for c in range(NUM_CLASSES)}
    for i, lbl in enumerate(ds["labels"]):
        by_class[lbl].append(i)
    picked = []
    for c, idxs in by_class.items():
        take = min(n_per_class, len(idxs))
        picked.extend(rng.choice(idxs, size=take, replace=False).tolist())
    return ds.select(sorted(picked))

eval_subset_long = stratified_subset(tokenized_long["validation"], n_per_class=100)
print(f"Eval subset: {len(eval_subset_long)} rows (~300, stratified by class)")

# Toggle off if the unsloth classifier-reload bug fires after load_best_model_at_end=True.
# When False, manual best-checkpoint selection runs in cell-11.
LOAD_BEST_FALLBACK = True

# 4096 tokens → batch=8 fits on T4. Effective batch = 8 * 4 = 32.
# 8K subsample / 32 = 250 steps (was 125 at 6144). logging_steps=5 + eval_steps=25
# gives 10 evals per epoch (must align with save_steps=25 when load_best_model_at_end=True).

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=tokenized_long["train"],
    eval_dataset=eval_subset_long,
    compute_metrics=compute_metrics,
    args=TrainingArguments(
        output_dir="./modernfinbert-v2-long-output",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=1e-4,
        weight_decay=0.001,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=SEED,
        group_by_length=True,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,
        logging_strategy="steps",
        logging_steps=5,
        load_best_model_at_end=LOAD_BEST_FALLBACK,
        metric_for_best_model="eval_macro_f1" if LOAD_BEST_FALLBACK else None,
        greater_is_better=True if LOAD_BEST_FALLBACK else None,
        report_to="none",
    ),
)

trainer.train()


## 4. Evaluation

In [ ]:
# Manual best-checkpoint selection (only runs if LOAD_BEST_FALLBACK=False)
if not LOAD_BEST_FALLBACK:
    import json, glob
    ckpt_dirs = sorted(glob.glob("./modernfinbert-v2-long-output/checkpoint-*"))
    best_ckpt, best_f1 = None, -1.0
    for d in ckpt_dirs:
        state = json.load(open(f"{d}/trainer_state.json"))
        for entry in state["log_history"]:
            f1 = entry.get("eval_macro_f1", -1)
            if f1 > best_f1:
                best_f1, best_ckpt = f1, d
    print(f"Best checkpoint: {best_ckpt} (eval_macro_f1={best_f1:.4f})")
    trainer._load_from_checkpoint(best_ckpt)

# Final eval on the full validation set (training used a subset for speed)
trainer.eval_dataset = tokenized_long["validation"]
full_val = trainer.evaluate()
print(f"Full val: acc={full_val['eval_accuracy']:.4f}, f1={full_val['eval_macro_f1']:.4f}")

# Evaluate on long-context test set
print("=" * 60)
print("LONG-CONTEXT TEST SET")
print("=" * 60)
long_results = trainer.evaluate(tokenized_long["test"])
print(f"Accuracy={long_results['eval_accuracy']:.4f}, Macro F1={long_results['eval_macro_f1']:.4f}")

preds_long = trainer.predict(tokenized_long["test"])
y_pred_long = preds_long.predictions.argmax(axis=-1)
y_true_long = preds_long.label_ids
print(classification_report(y_true_long, y_pred_long, target_names=LABEL_NAMES))

# Evaluate on medium-context test set (regression check)
print("=" * 60)
print("MEDIUM-CONTEXT TEST SET (regression check)")
print("=" * 60)
med_results = trainer.evaluate(tokenized_med_test)
print(f"Accuracy={med_results['eval_accuracy']:.4f}, Macro F1={med_results['eval_macro_f1']:.4f}")

preds_med = trainer.predict(tokenized_med_test)
y_pred_med = preds_med.predictions.argmax(axis=-1)
y_true_med = preds_med.label_ids
print(classification_report(y_true_med, y_pred_med, target_names=LABEL_NAMES))

# Evaluate on short-context test set (regression check)
print("=" * 60)
print("SHORT-CONTEXT TEST SET (regression check)")
print("=" * 60)
short_results = trainer.evaluate(tokenized_short_test)
print(f"Accuracy={short_results['eval_accuracy']:.4f}, Macro F1={short_results['eval_macro_f1']:.4f}")

preds_short = trainer.predict(tokenized_short_test)
y_pred_short = preds_short.predictions.argmax(axis=-1)
y_true_short = preds_short.label_ids
print(classification_report(y_true_short, y_pred_short, target_names=LABEL_NAMES))

print("\nTraining log:")
for entry in trainer.state.log_history:
    if "eval_loss" in entry:
        print(f"  Epoch {entry.get('epoch', '?')}: loss={entry['eval_loss']:.4f}, "
              f"acc={entry.get('eval_accuracy', 0):.4f}, f1={entry.get('eval_macro_f1', 0):.4f}")

## 5. Push Final Model to HuggingFace

In [ ]:
merged = model.merge_and_unload()
merged.push_to_hub(HF_PUSH_NAME, private=False)
tokenizer.push_to_hub(HF_PUSH_NAME, private=False)
print(f"Final model pushed to {HF_PUSH_NAME}")